# Agentic Market Research & Trend Analysis Tutorial

A compact notebook that runs an end-to-end agentic research workflow with:
- `openai`
- `openai-agents`
- `requests`

## Design the agentic research workflow

### Stage 1: Quick Web-Grounded Snapshot
Use the Answer endpoint to collect a focused market summary and source URLs.

### Stage 2: Source Expansion
Select the top 3 sources and scrape them for deeper context.

### Stage 3: Signal Extraction
Extract recurring use cases, positioning, and feature patterns.

### Stage 4: Trend Synthesis
Convert individual signals into higher-level trends and confidence signals.

### Stage 5: Brief Generation
Generate a concise technical research brief and save it as markdown.


### Setup

### Environment
Assumes `OPENAI_API_KEY` and `OLOSTEP_API_KEY` are already configured.


In [1]:
from __future__ import annotations

import json
import os
from typing import Any

import requests
from openai import AsyncOpenAI
from agents import Agent, RunConfig, Runner, function_tool, set_default_openai_client

MODEL_NAME = "gpt-5.2"
INITIAL_TASK = (
    "Research current trends in AI agent tools used by SMB marketing teams. "
    "Focus on recurring use cases, positioning, and common feature patterns."
)
OLOSTEP_BASE_URL = os.getenv("OLOSTEP_BASE_URL", "https://api.olostep.com").rstrip("/")
BRIEF_PATH = "agents_sdk_style_market_research_top3_brief.md"
RESULT_PATH = "agents_sdk_style_market_research_top3_result.json"

set_default_openai_client(AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"]))

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {os.environ['OLOSTEP_API_KEY']}",
    "Content-Type": "application/json",
})


## Integrate the Olostep APIs for answer and scraping

In [2]:
def parse_json_object(value: Any) -> dict[str, Any]:
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("```"):
            text = text.strip("`").replace("json", "", 1).strip()
        return json.loads(text)
    return {}


def unique_http_urls(items: list[Any]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for item in items:
        url = str(item).strip()
        if url.startswith(("http://", "https://")) and url not in seen:
            seen.add(url)
            out.append(url)
    return out


def compact_text(value: Any, limit: int = 7000) -> str:
    return str(value or "").strip()[:limit]


def request_olostep(path: str, payload: dict[str, Any]) -> dict[str, Any]:
    response = session.post(f"{OLOSTEP_BASE_URL}{path}", json=payload, timeout=60)
    response.raise_for_status()
    return response.json()


@function_tool
def olostep_answer_tool(task: str) -> dict[str, Any]:
    """Call Olostep Answer API."""
    return request_olostep("/v1/answers", {"task": task})


@function_tool
def olostep_scrape_tool(url: str) -> dict[str, Any]:
    """Call Olostep Scrape API for one URL."""
    return request_olostep("/v1/scrapes", {"url_to_scrape": url, "formats": ["markdown", "text"]})


## Build a GPT-5.2 research agent with OpenAI Agents SDK

### Agent responsibilities
- `research_agent`: answer + top source scraping package
- `extraction_agent`: structured signal extraction
- `trend_agent`: trend analysis from signals
- `brief_agent`: concise technical brief


In [3]:
research_agent = Agent(
    name="research_agent",
    model=MODEL_NAME,
    tools=[olostep_answer_tool, olostep_scrape_tool],
    instructions=(
        "Always keep INITIAL_TASK central.\n"
        "Run this exact flow:\n"
        "1) Call olostep_answer_tool once with INITIAL_TASK.\n"
        "2) Parse result.json_content and result.sources.\n"
        "3) Select top 3 unique URLs (prefer result.sources order).\n"
        "4) Scrape those top 3 URLs with olostep_scrape_tool.\n"
        "Return strict JSON only with keys: initial_task, answer_summary, answer_json_content, answer_sources, top3_sources, scraped_pages."
    ),
)

extraction_agent = Agent(
    name="extraction_agent",
    model=MODEL_NAME,
    instructions=(
        "Always include INITIAL_TASK context.\n"
        "Extract concrete market signals from provided summary + scraped context only.\n"
        "Return strict JSON with: signals (list of objects).\n"
        "Each signal object: topic, use_case, positioning_pattern, feature_pattern, evidence, source_url."
    ),
)

trend_agent = Agent(
    name="trend_agent",
    model=MODEL_NAME,
    instructions=(
        "Always include INITIAL_TASK context.\n"
        "Analyze recurring patterns from extracted signals.\n"
        "Return strict JSON with: trends (list) and summary (string).\n"
        "Each trend object: trend, why_now, supporting_signals, source_urls, confidence_0_to_1."
    ),
)

brief_agent = Agent(
    name="brief_agent",
    model=MODEL_NAME,
    instructions=(
        "Always include INITIAL_TASK context.\n"
        "Write a concise technical research brief in markdown.\n"
        "Use sections: Executive Summary, Top Trends, Recurring Use Cases, Positioning Patterns, Feature Patterns, Sources."
    ),
)


## Running Market Research Agent


In [4]:
research_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Use tools to complete the flow exactly and return strict JSON only.
"""

research_run = await Runner.run(
    research_agent,
    input=research_prompt,
    run_config=RunConfig(model=MODEL_NAME),
)
research_payload = parse_json_object(research_run.final_output)

print("Top sources:")
print(
    json.dumps(research_payload.get("top3_sources", []), indent=2, ensure_ascii=False)
)


Top sources:
[
  "https://blogs.microsoft.com/blog/2025/04/28/how-agentic-ai-is-driving-ai-first-business-transformation-for-customers-to-achieve-more/",
  "https://www.emarketer.com/content/2024--year-ai-agents-transformed-chatbots-productivity-powerhouses",
  "https://www.salesforce.com/en-us/wp-content/uploads/sites/4/documents/resources/smb-trends-report-6th-edition_Salesforce.pdf"
]


## Extract market signals and analyze trends
Extract signals first, then synthesize trends.


In [5]:
extraction_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Extract signals from this research package. Return strict JSON only.

RESEARCH_PACKAGE:
{json.dumps(research_payload, ensure_ascii=False)}
"""

extraction_run = await Runner.run(
    extraction_agent,
    input=extraction_prompt,
    run_config=RunConfig(model=MODEL_NAME),
)
extraction_payload = parse_json_object(extraction_run.final_output)
signals = extraction_payload.get("signals", [])

print("Signals extracted:", len(signals))
print(signals[:500])

Signals extracted: 13
[{'topic': 'No/low-code agent builders plus prebuilt agents for business workflows', 'use_case': 'SMB teams create custom agents and deploy prebuilt agents to automate common work (e.g., sales/research) inside existing collaboration tools', 'positioning_pattern': 'Autonomy + easy creation (no/low-code) + measurable productivity/cost outcomes + governance', 'feature_pattern': 'No/low-code agent studio; library of prebuilt agents; workflow automation; native integration into collaboration apps (e.g., Teams); measurement of impact; governance controls', 'evidence': 'Microsoft emphasizes “no-code/low-code creation of custom agents in Copilot Studio” and “prebuilt agents (e.g., Sales Agent, Researcher/Analyst)” that “automate workflows, integrate into tools like Teams, and provide measurable productivity/cost improvements,” reflecting “autonomy + integration + governance.”', 'source_url': 'https://blogs.microsoft.com/blog/2025/04/28/how-agentic-ai-is-driving-ai-first-b

In [6]:
trend_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Run trend analysis from the extracted signals. Return strict JSON only.

SIGNALS_JSON:
{json.dumps(signals, ensure_ascii=False)}
"""

trend_run = await Runner.run(
    trend_agent,
    input=trend_prompt,
    run_config=RunConfig(model=MODEL_NAME),
)
trend_payload = parse_json_object(trend_run.final_output)
trends = trend_payload.get("trends", [])

print("Trends identified:", len(trends))
print(trends[:500])

Trends identified: 11
[{'trend': 'No/low-code agent builders plus libraries of prebuilt marketing/sales agents', 'why_now': 'SMB marketing teams want automation without hiring engineering, and vendors can now package repeatable “agent” workflows with templates while still offering customization. Mature copilots/agent studios make governance and deployment in existing tools feasible.', 'supporting_signals': ['No/low-code agent studio + prebuilt agents for business workflows; integrations into collaboration tools; measurable productivity/cost outcomes; governance controls (Microsoft Copilot Studio + prebuilt agents).', 'Agentic tools positioned as end-to-end assistants for lean SMB teams with playbooks/templates and SMB-friendly setup.'], 'source_urls': ['https://blogs.microsoft.com/blog/2025/04/28/how-agentic-ai-is-driving-ai-first-business-transformation-for-customers-to-achieve-more/', 'RESEARCH_PACKAGE.answer_summary'], 'confidence_0_to_1': 0.86}, {'trend': 'Shift from chat-style ass

## Generate the technical research brief
Generate the brief, save markdown + json, and preview output.


In [7]:
brief_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Generate the final concise technical research brief in markdown.

ANSWER_SUMMARY_AND_CONTEXT:
{json.dumps({
    "answer_summary": research_payload.get("answer_summary", ""),
    "top3_sources": research_payload.get("top3_sources", []),
    "scraped_pages": research_payload.get("scraped_pages", []),
}, ensure_ascii=False)}

EXTRACTED_SIGNALS:
{json.dumps(signals, ensure_ascii=False)}

TREND_ANALYSIS:
{json.dumps(trends, ensure_ascii=False)}
"""

brief_run = await Runner.run(
    brief_agent,
    input=brief_prompt,
    run_config=RunConfig(model=MODEL_NAME),
)
final_brief = str(brief_run.final_output).strip()

result = {
    "initial_task": INITIAL_TASK,
    "model": MODEL_NAME,
    "research_payload": research_payload,
    "signals": signals,
    "trends": trends,
    "brief_markdown": final_brief,
}

with open(BRIEF_PATH, "w", encoding="utf-8") as f:
    f.write(final_brief + "\n")

with open(RESULT_PATH, "w", encoding="utf-8") as f:
    f.write(json.dumps(result, indent=2, ensure_ascii=False) + "\n")

print(f"Saved brief to: {BRIEF_PATH}")
print(f"Saved result to: {RESULT_PATH}")
print("\n--- Brief Preview ---\n")
print(final_brief[:3000])


Saved brief to: agents_sdk_style_market_research_top3_brief.md
Saved result to: agents_sdk_style_market_research_top3_result.json

--- Brief Preview ---

# AI Agent Tools for SMB Marketing Teams (2024–2025) — Technical Research Brief

## Executive Summary
SMB marketing teams are rapidly adopting **agentic AI tools** positioned as **autonomous or semi-autonomous “force multipliers”** that can execute multi-step work (plan → create → launch → optimize → report) rather than only generate content or answer questions. The dominant product pattern is a **closed-loop system**: ingest real-time signals, decide/plan, take actions inside the marketing/sales stack, and measure lift with feedback loops. Differentiation is increasingly driven by (1) **no/low-code agent builders + prebuilt playbooks**, (2) **deep integrations** into common SMB tools, and (3) **trust/safety controls** (approvals, spend caps, auditability) as agents begin modifying budgets, messaging, and customer touches.

## Top Tre

### Workflow Recap and What the Result Means

### Recap
This notebook executes a single agentic pipeline:
1. gather market snapshot and sources,
2. scrape the top sources,
3. extract signals,
4. synthesize trends,
5. produce a concise technical brief.

### Result Objects
- `research_payload`: source-grounded context package from the research stage
- `signals`: atomic evidence points extracted from summary + scraped pages
- `trends`: higher-level patterns synthesized from signal clusters
- `brief_markdown`: final narrative report for decision-making and sharing

### Saved Files
- `agents_sdk_style_market_research_top3_brief.md`: final readable brief
- `agents_sdk_style_market_research_top3_result.json`: full structured output from all stages
